In [ ]:
def calculate_frequencies(start_freq, num_steps=10, num_octaves=1, precision=2):
    """
    Calculate frequencies dividing octaves into equal steps.
    
    Parameters:
    start_freq (float): Starting frequency in Hz
    num_steps (int): Number of steps to divide each octave into (default: 10)
    num_octaves (float): Number of octaves to calculate (default: 1)
    precision (int): Number of decimal places for rounding (default: 2)
    
    Returns:
    list: List of frequencies including start and end frequencies
    """
    # Calculate total steps needed
    total_steps = int(num_steps * num_octaves)
    
    # Calculate the multiplication factor for each step
    # For multiple octaves, we need to reach 2^num_octaves
    target_multiplier = 2 ** num_octaves
    step_factor = target_multiplier ** (1/total_steps)
    
    # Initialize list with starting frequency
    frequencies = []
    current_freq = start_freq
    
    # Calculate each frequency including the final frequency
    for _ in range(total_steps + 1):
        frequencies.append(round(current_freq, precision))
        current_freq *= step_factor
        
    return frequencies

# Example usage:
# Single octave with 3 decimal places
# frequencies = calculate_frequencies(440, precision=3)

# Two octaves
# frequencies = calculate_frequencies(440, num_octaves=2)

# Half octave with 1 decimal place
# frequencies = calculate_frequencies(440, num_octaves=0.5, precision=1)

In [4]:
# pip install numpy sounddevice

Note: you may need to restart the kernel to use updated packages.


In [5]:
import numpy as np
import sounddevice as sd

def play_frequencies(frequencies, duration=0.5, amplitude=0.3, sample_rate=44100):
    """
    Play a sequence of frequencies as sine waves.
    
    Parameters:
    frequencies (list): List of frequencies in Hz to play
    duration (float): Duration of each tone in seconds
    amplitude (float): Volume of the tone (0.0 to 1.0)
    sample_rate (int): Audio sample rate in Hz
    """
    # Create time array
    t = np.linspace(0, duration, int(sample_rate * duration), False)
    
    # Add small fade in/out to avoid clicking
    fade_duration = 0.01  # 10ms fade
    fade_length = int(fade_duration * sample_rate)
    fade_in = np.linspace(0, 1, fade_length)
    fade_out = np.linspace(1, 0, fade_length)
    
    # Play each frequency
    for freq in frequencies:
        # Generate sine wave
        tone = amplitude * np.sin(2 * np.pi * freq * t)
        
        # Apply fade in/out
        tone[:fade_length] *= fade_in
        tone[-fade_length:] *= fade_out
        
        # Play the tone
        sd.play(tone, sample_rate)
        sd.wait()  # Wait until the sound has finished playing

# Example usage:
if __name__ == "__main__":
    # Example frequencies (A4 to A5 in 10 steps)
#     freqs = [440.00, 471.64, 505.53, 541.83, 580.71, 
#              622.35, 666.95, 714.71, 765.84, 820.57, 880.00]
    
    freqs = calculate_frequencies(start_freq=440, num_steps=10, num_octaves=1, precision=2)
    
    # Play each frequency for 0.5 seconds
    play_frequencies(freqs)

    # # Alternative usage with different parameters:
    # play_frequencies(freqs, duration=0.3, amplitude=0.5)

In [10]:
import math

def get_pi(n):
    """
    Returns the first n decimal places of pi.
    
    Parameters:
    n (int): Number of decimal places desired
    
    Returns:
    list: List of integers representing each decimal place of pi
    """
    # Get pi to sufficient precision and convert to string
    pi_str = str(math.pi)
    
    # Take integer part (3) and n decimal places
    pi_digits = pi_str[0] + pi_str[2:n+2]
    
    # Convert to list of integers
    return [int(d) for d in pi_digits]



In [18]:
def map_numbers_to_frequencies(numbers, frequencies):
    """
    Maps single digits to frequency indices.
    
    Parameters:
    numbers (list): List of integers 0-9
    frequencies (list): List of frequencies
    
    Returns:
    dict: Dictionary mapping each number to its corresponding frequency index
    """
    # Validate input numbers are 0-9
    if not all(0 <= num <= 9 for num in numbers):
        raise ValueError("All numbers must be between 0 and 9")
        
    # Create mapping dictionary
    mapping = {num: frequencies[idx] for idx, num in enumerate(numbers)}
    
    return mapping

freqs = [440.00, 471.64, 505.53, 541.83, 580.71, 
             622.35, 666.95, 714.71, 765.84, 820.57]

# Define the decimal value --> frequency mapping dict first
freq_dict = map_numbers_to_frequencies(list(range(10)), freqs)

In [20]:
print(freq_dict)
print(get_pi(10))


{0: 440.0, 1: 471.64, 2: 505.53, 3: 541.83, 4: 580.71, 5: 622.35, 6: 666.95, 7: 714.71, 8: 765.84, 9: 820.57}
[3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5]


In [ ]:
import numpy as np
import sounddevice as sd

def play_frequencies(frequencies, duration=0.1, amplitude=0.3, sample_rate=44100):
    """
    Play a sequence of frequencies as sine waves with smooth transitions,
    optimized for quick playback.
    
    Parameters:
    frequencies (list): List of frequencies in Hz to play
    duration (float): Duration of each tone in seconds
    amplitude (float): Volume of the tone (0.0 to 1.0)
    sample_rate (int): Audio sample rate in Hz
    """
    # Create time array
    t = np.linspace(0, duration, int(sample_rate * duration), False)
    
    # Make fade duration proportional to note duration, with minimum and maximum limits
    fade_duration = min(duration * 0.15, 0.015)  # 15% of duration, max 15ms
    fade_length = int(fade_duration * sample_rate)
    
    # Ensure fade_length is at least 1 sample
    fade_length = max(1, fade_length)
    
    # Create smoother fade curves using half-cosine
    fade_in = np.cos(np.linspace(np.pi, 2*np.pi, fade_length)) * 0.5 + 0.5
    fade_out = np.cos(np.linspace(0, np.pi, fade_length)) * 0.5 + 0.5
    
    # Play each frequency
    for freq in frequencies:
        # Generate sine wave
        tone = amplitude * np.sin(2 * np.pi * freq * t)
        
        # Apply fades
        envelope = np.ones_like(t)
        envelope[:fade_length] *= fade_in
        envelope[-fade_length:] *= fade_out
        
        # Apply envelope to tone
        tone *= envelope
        
        # Add tiny silence (0.5ms) between notes to prevent overlap
        silence = np.zeros(int(0.0005 * sample_rate))
        tone_with_silence = np.concatenate([tone, silence])
        
        # Play the tone
        sd.play(tone_with_silence, sample_rate)
        sd.wait()

# Example usage:
if __name__ == "__main__":
    # Example frequencies (A4 to A5 in 10 steps)
    freqs = [440.00, 471.64, 505.53, 541.83, 580.71, 
             622.35, 666.95, 714.71, 765.84, 820.57, 880.00]
    
    # Play each frequency for 0.1 seconds
    play_frequencies(freqs, duration=0.2)
    

In [44]:
# Play notes quickly
play_frequencies(freqs, duration=0.1)  # 100ms per note
play_frequencies(freqs, duration=0.05)  # 50ms per note
play_frequencies(freqs, duration=0.03)  # 30ms per note

In [ ]:
# Example usage:
if __name__ == "__main__":
    # Example frequencies (A4 to A5 in 10 steps)
    pi_freqs = [freq_dict[n] for n in get_pi(10)]
    play_frequencies(pi_freqs, duration=0.5, amplitude=0.3, sample_rate=44100)



In [45]:
import numpy as np
import sounddevice as sd

def play_frequencies(frequencies, duration=0.1, amplitude=0.5, sample_rate=44100):
    """
    Play a sequence of frequencies as sine waves with minimal transitions.
    
    Parameters:
    frequencies (list): List of frequencies in Hz to play
    duration (float): Duration of each tone in seconds (minimum 0.03s recommended)
    amplitude (float): Volume of the tone (0.0 to 1.0)
    sample_rate (int): Audio sample rate in Hz
    """
    # Ensure minimum duration
    duration = max(0.03, duration)
    
    # Create time array
    t = np.linspace(0, duration, int(sample_rate * duration), False)
    
    # Very short fade duration (1% of note duration or 1ms, whichever is smaller)
    fade_duration = min(duration * 0.01, 0.001)  # 1ms maximum
    fade_length = max(10, int(fade_duration * sample_rate))  # minimum 10 samples
    
    # Simple linear fades
    fade_in = np.linspace(0, 1, fade_length)
    fade_out = np.linspace(1, 0, fade_length)
    
    # Play each frequency
    for freq in frequencies:
        # Generate sine wave
        tone = amplitude * np.sin(2 * np.pi * freq * t)
        
        # Apply minimal fades
        tone[:fade_length] *= fade_in
        tone[-fade_length:] *= fade_out
        
        # Add minimal silence (0.1ms)
        silence = np.zeros(int(0.0001 * sample_rate))
        tone_with_silence = np.concatenate([tone, silence])
        
        # Play the tone
        sd.play(tone_with_silence, sample_rate)
        sd.wait()

# Example usage:
if __name__ == "__main__":
    # Example frequencies (A4 to A5 in 10 steps)
    freqs = [440.00, 471.64, 505.53, 541.83, 580.71, 
             622.35, 666.95, 714.71, 765.84, 820.57, 880.00]
    
    # Play each frequency
    play_frequencies(freqs, duration=0.03)

In [13]:

# Example usage:
if __name__ == "__main__":
    # Get first 10 decimal places of pi
    pi_decimals = get_pi(10)
    print(f"First 10 decimals of pi: {pi_decimals}")
    
    # Example with numbers 0-9 in order
    numbers = list(range(10))
    freqs = [440.00, 471.64, 505.53, 541.83, 580.71, 
             622.35, 666.95, 714.71, 765.84, 820.57]
             
    mapping = map_numbers_to_frequencies(numbers, freqs)
    print(mapping)
#     print("\nNumber to frequency index mapping:")
#     for num, idx in mapping.items():
#         print(f"Number {num} -> Index {idx} (Frequency: {freqs[idx]:.2f} Hz)")

First 10 decimals of pi: [3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5]
{0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9}
